In [ ]:

import pandas as pd

human = pd.read_csv('../data/random_essays_50.csv', sep='\t', encoding='utf-8', on_bad_lines='skip')
human['domain2_score'].fillna(0, inplace=True)
human_score = human['domain1_score'] + human['domain2_score']
essay_id = human['essay_id']
essay_set = human['essay_set']
for i in range(0,50):
    if essay_set[i] == 1:
        human_score[i] = human_score[i]/12
    elif essay_set[i] == 2:
        human_score[i] = human_score[i]/10
    elif essay_set[i] == 7:
        human_score[i] = human_score[i]/30
    elif essay_set[i] == 8:
        human_score[i] = human_score[i]/60
human_score = round(human_score, 3)

debate = pd.read_csv('../output/AES/debate/0.feedbacks.csv', sep='\t', encoding='utf-8', on_bad_lines='skip')
debate_score = debate['score']

baseline = pd.read_csv('../output/AES/baseline/0.feedbacks.csv', sep='\t', encoding='utf-8', on_bad_lines='skip')
baseline_score = baseline['score']

ma = pd.read_csv('../output/AES/ma/0.feedbacks.csv', sep='\t', encoding='utf-8', on_bad_lines='skip')
ma_score = ma['score']

bleu = pd.read_csv('../data/bleu_scores.csv', sep='\t', encoding='utf-8', on_bad_lines='skip')
bleu1_score = bleu['bleu1_score']
bleu2_score = bleu['bleu2_score']
bleu4_score = bleu['bleu4_score']

rouge = pd.read_csv('../data/rouge_scores.csv', sep='\t', encoding='utf-8', on_bad_lines='skip')
rouge1_score = rouge['rouge1']
rouge2_score = rouge['rouge2']
rougeL_score = rouge['rougeL']

distinct = pd.read_csv('../data/distinct_scores.csv', sep='\t', encoding='utf-8', on_bad_lines='skip')
distinct1_score = distinct['distinct_1']
distinct2_score = distinct['distinct_2']
distinct4_score = distinct['distinct_4']

bert = pd.read_csv('../data/bert_scores.csv', sep='\t', encoding='utf-8', on_bad_lines='skip')
bert_score = bert['bert_finetuned_score']


/var/folders/7l/9vhlxqcx2zbbdx1gdlkfm1f40000gn/T/ipykernel_22029/1330669852.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  human['domain2_score'].fillna(0, inplace=True)


In [40]:
# 将所有数据合并到一个DataFrame中
data = pd.DataFrame({
    'essay_id': essay_id,
    'human': human_score,
    'debate': debate_score,
    'baseline': baseline_score,
    'ma': ma_score,
    'bleu1': bleu1_score,
    'bleu2': bleu2_score,
    'bleu4': bleu4_score,
    'rouge1': rouge1_score,
    'rouge2': rouge2_score,
    'rougeL': rougeL_score,
    'distinct1': distinct1_score,
    'distinct2': distinct2_score,
    'distinct4': distinct4_score,
    'bert': bert_score
})
data.to_csv('../data/all_score.csv', sep='\t', index=False, encoding='utf-8')

essay_set = 1:
domain1_score 满分：

essay_set = 2:
domain1_score + domain2_score 

essay_set = 7:
domain1_score

In [57]:
# QWK
from sklearn.metrics import cohen_kappa_score
import numpy as np

def confusion_matrix(rater_a, rater_b, min_rating=None, max_rating=None):
    """
    Returns the confusion matrix between rater's ratings
    """
    assert(len(rater_a) == len(rater_b))
    if min_rating is None:
        min_rating = min(rater_a + rater_b)
    if max_rating is None:
        max_rating = max(rater_a + rater_b)
    num_ratings = int(max_rating - min_rating + 1)
    conf_mat = [[0 for i in range(num_ratings)]
                for j in range(num_ratings)]
    for a, b in zip(rater_a, rater_b):
        conf_mat[a - min_rating][b - min_rating] += 1
    return conf_mat

def histogram(ratings, min_rating=None, max_rating=None):
    """
    Returns the counts of each type of rating that a rater made
    """
    if min_rating is None:
        min_rating = min(ratings)
    if max_rating is None:
        max_rating = max(ratings)
    num_ratings = int(max_rating - min_rating + 1)
    hist_ratings = [0 for x in range(num_ratings)]
    for r in ratings:
        hist_ratings[r - min_rating] += 1
    return hist_ratings

def quadratic_weighted_kappa(rater_a, rater_b, min_rating=None, max_rating=None):
    """
    Calculates the quadratic weighted kappa
    quadratic_weighted_kappa calculates the quadratic weighted kappa
    value, which is a measure of inter-rater agreement between two raters
    that provide discrete numeric ratings.  Potential values range from -1
    (representing complete disagreement) to 1 (representing complete
    agreement).  A kappa value of 0 is expected if all agreement is due to
    chance.

    quadratic_weighted_kappa(rater_a, rater_b), where rater_a and rater_b
    each correspond to a list of integer ratings.  These lists must have the
    same length.

    The ratings should be integers, and it is assumed that they contain
    the complete range of possible ratings.

    quadratic_weighted_kappa(X, min_rating, max_rating), where min_rating
    is the minimum possible rating, and max_rating is the maximum possible
    rating
    """
    rater_a = np.array(rater_a, dtype=int)
    rater_b = np.array(rater_b, dtype=int)
    assert(len(rater_a) == len(rater_b))
    if min_rating is None:
        min_rating = min(min(rater_a), min(rater_b))
    if max_rating is None:
        max_rating = max(max(rater_a), max(rater_b))
    conf_mat = confusion_matrix(rater_a, rater_b,
                                min_rating, max_rating)
    num_ratings = len(conf_mat)
    num_scored_items = float(len(rater_a))

    hist_rater_a = histogram(rater_a, min_rating, max_rating)
    hist_rater_b = histogram(rater_b, min_rating, max_rating)

    numerator = 0.0
    denominator = 0.0

    for i in range(num_ratings):
        for j in range(num_ratings):
            expected_count = (hist_rater_a[i] * hist_rater_b[j]
                              / num_scored_items)
            d = pow(i - j, 2.0) / pow(num_ratings - 1, 2.0)
            numerator += d * conf_mat[i][j] / num_scored_items
            denominator += d * expected_count / num_scored_items

    return 1.0 - numerator / denominator

human = pd.read_csv('../data/random_essays_50.csv', sep='\t', encoding='utf-8', on_bad_lines='skip')
rater_a = human['domain1_score']
rater_a = [int(i) for i in rater_a.tolist()]

In [59]:
data = pd.read_csv('../data/all_score.csv', sep='\t', encoding='utf-8', on_bad_lines='skip')

corr = pd.DataFrame(columns=['debate', 'baseline', 'ma', 'bleu1', 'bleu2', 'bleu4', 'rouge1', 'rouge2', 'rougeL', 'distinct1', 'distinct2', 'distinct4', 'bert'])
corr_p = pd.DataFrame(columns=['debate', 'baseline', 'ma', 'bleu1', 'bleu2', 'bleu4', 'rouge1', 'rouge2', 'rougeL', 'distinct1', 'distinct2', 'distinct4', 'bert'])


# 计算Pearson相关系数
from scipy.stats import pearsonr
for col in corr.columns:
    r, p = pearsonr(human_score, data[col])
    corr.loc['pearson', col] = r  # 存储 Pearson r
    corr_p.loc['pearson', col] = p  # 存储 p 值

# 计算Spearman相关系数
from scipy.stats import spearmanr
for col in corr.columns:
    r, p = spearmanr(human_score, data[col])
    corr.loc['spearman', col] = r  # 存储 Spearman r
    corr_p.loc['spearman', col] = p  # 存储 p 值

# 计算Kendall相关系数
from scipy.stats import kendalltau
for col in corr.columns:
    r, p = kendalltau(human_score, data[col])
    corr.loc['kendall', col] = r  # 存储 Kendall r
    corr_p.loc['kendall', col] = p  # 存储 p 值

for col in corr.columns:
    rater = data[col] * 12
    rater = [int(i) for i in rater.tolist()]
    r = quadratic_weighted_kappa(human_score, rater)
    corr.loc['qwk', col] = r  # 存储 QWK

print(corr)

            debate  baseline        ma     bleu1     bleu2     bleu4  \
pearson   0.323872  0.690985  0.645471  0.807904  0.779903  0.416893   
spearman  0.328762  0.570064  0.578628  0.742507  0.730153  0.500988   
kendall   0.281501  0.444998  0.461048  0.566904  0.574178  0.377179   
qwk       0.000625  0.001721  0.001202  0.011692  0.059517 -0.027397   

            rouge1    rouge2    rougeL distinct1 distinct2 distinct4      bert  
pearson   0.806962  0.517337  0.675039  -0.58547   -0.2596 -0.124497  0.130699  
spearman  0.710827   0.54955  0.612424  -0.38441 -0.202313 -0.172194  0.004552  
kendall   0.549319   0.40259  0.465434 -0.282802 -0.145044 -0.117953  0.002512  
qwk       0.004968 -0.066351 -0.002506  -0.00018 -0.000235 -0.000128  0.000198  


In [33]:
rater_debate = debate['score']
rater_debate = rater_debate * 12
rater_debate = [int(i) for i in rater_debate.tolist()]
qwk_debate = quadratic_weighted_kappa(rater_a, rater_debate)

rater_baseline = baseline['score']
rater_baseline = rater_baseline * 12
rater_baseline = [int(i) for i in rater_baseline.tolist()]
qwk_baseline = quadratic_weighted_kappa(rater_a, rater_baseline)

rater_ma = ma['score']
rater_ma = rater_ma * 12
rater_ma = [int(i) for i in rater_ma.tolist()]
qwk_ma = quadratic_weighted_kappa(rater_a, rater_ma)

rater_bleu1 = bleu['bleu1_score']
rater_bleu1 = rater_bleu1 * 12
rater_bleu1 = [int(i) for i in rater_bleu1.tolist()]
qwk_bleu1 = quadratic_weighted_kappa(rater_a, rater_bleu1)

rater_bleu2 = bleu['bleu2_score']
rater_bleu2 = rater_bleu2 * 12
rater_bleu2 = [int(i) for i in rater_bleu2.tolist()]
qwk_bleu2 = quadratic_weighted_kappa(rater_a, rater_bleu2)

rater_bleu4 = bleu['bleu4_score']
rater_bleu4 = rater_bleu4 * 12
rater_bleu4 = [int(i) for i in rater_bleu4.tolist()]
qwk_bleu4 = quadratic_weighted_kappa(rater_a, rater_bleu4)

rater_rouge1 = rouge['rouge1']
rater_rouge1 = rater_rouge1 * 12
rater_rouge1 = [int(i) for i in rater_rouge1.tolist()]
qwk_rouge1 = quadratic_weighted_kappa(rater_a, rater_rouge1)

rater_rouge2 = rouge['rouge2']
rater_rouge2 = rater_rouge2 * 12
rater_rouge2 = [int(i) for i in rater_rouge2.tolist()]
qwk_rouge2 = quadratic_weighted_kappa(rater_a, rater_rouge2)

rater_rougeL = rouge['rougeL']
rater_rougeL = rater_rougeL * 12
rater_rougeL = [int(i) for i in rater_rougeL.tolist()]
qwk_rougeL = quadratic_weighted_kappa(rater_a, rater_rougeL)

rater_distinct1 = distinct['distinct_1']
rater_distinct1 = rater_distinct1 * 12
rater_distinct1 = [int(i) for i in rater_distinct1.tolist()]
qwk_distinct1 = quadratic_weighted_kappa(rater_a, rater_distinct1)

rater_distinct2 = distinct['distinct_2']
rater_distinct2 = rater_distinct2 * 12
rater_distinct2 = [int(i) for i in rater_distinct2.tolist()]
qwk_distinct2 = quadratic_weighted_kappa(rater_a, rater_distinct2)

rater_distinct4 = distinct['distinct_4']
rater_distinct4 = rater_distinct4 * 12
rater_distinct4 = [int(i) for i in rater_distinct4.tolist()]
qwk_distinct4 = quadratic_weighted_kappa(rater_a, rater_distinct4)

rater_bert = bert['bert_finetuned_score']
rater_bert = rater_bert * 12
rater_bert = [int(i) for i in rater_bert.tolist()]
qwk_bert = quadratic_weighted_kappa(rater_a, rater_bert)

new_row = pd.DataFrame({'debate': [qwk_debate], 'baseline': [qwk_baseline], 'ma': [qwk_ma], 'bleu1': [qwk_bleu1], 'bleu2': [qwk_bleu2], 'bleu4': [qwk_bleu4], 'rouge1': [qwk_rouge1], 'rouge2': [qwk_rouge2], 'rougeL': [qwk_rougeL], 'distinct1': [qwk_distinct1], 'distinct2': [qwk_distinct2], 'distinct4': [qwk_distinct4], 'bert': [qwk_bert]})
corr = pd.concat([corr, new_row])
corr.index = ['pearson', 'kendall', 'spearman', 'QWK']
print(corr)

            debate  baseline        ma     bleu1    bleu2     bleu4    rouge1  \
pearson   0.324000  0.691000  0.645000  0.808000  0.78000  0.417000  0.807000   
kendall   0.282000  0.445000  0.461000  0.567000  0.57400  0.377000  0.549000   
spearman  0.329000  0.570000  0.579000  0.743000  0.73000  0.501000  0.711000   
QWK       0.021417  0.087991  0.068452  0.006787 -0.00005 -0.000571  0.005064   

            rouge2    rougeL  distinct1  distinct2  distinct4     bert  
pearson   0.517000  0.675000  -0.585000  -0.260000  -0.124000  0.13100  
kendall   0.403000  0.465000  -0.283000  -0.145000  -0.118000  0.00300  
spearman  0.550000  0.612000  -0.384000  -0.202000  -0.172000  0.00500  
QWK      -0.004588  0.000528  -0.013669  -0.003767  -0.002614 -0.04941  


In [27]:
# 保存结果
# 转置
corr = corr.T


In [28]:
# 保存到excel
corr.to_excel('/Users/ylm/THU/code/exp/data/correlation.xlsx', sheet_name='correlation_new')